# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook provides a step-by-step guide for loading and exploring the FAIR\^2 dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library. The notebook demonstrates how to access metadata, enumerate record sets, extract fields by their `@id`, and perform exploratory analysis and visualization workflows.

### Dataset Source
The dataset is defined by a Croissant schema accessible from the following URL:

[https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -q mlcroissant

## 1. Data Loading

Load dataset metadata using `mlcroissant`. We will access the dataset object and print a short description.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset name: {metadata.name}\nDescription: {metadata.description}")
print(f"Version: {metadata.version}\nLicense: {metadata.license}")

## 2. Data Overview

Enumerate available `RecordSet` entities and their associated `Field` definitions. All are referenced by their unique `@id`. This allows us to see what record sets and columns are available for further analysis.

In [ ]:
# List all record sets (@id and name) in the dataset schema

print("Available RecordSets (by @id):")
for rs in metadata.record_sets:
    print(f"  - @id: {rs.id} | name: {rs.name}")

# For each record set, print its available fields (by @id and name)
print("\nFields for each RecordSet:")
for rs in metadata.record_sets:
    print(f"- RecordSet {rs.id}:")
    for field in rs.fields:
        col_ids = [col.id for col in getattr(field, 'columns', [])]
        print(f"    Field @id: {field.id} | name: {field.name} | columns: {col_ids}")

## 3. Data Extraction

Load data from a selected `RecordSet` into a DataFrame for analysis. The `RecordSet` and `Field` `@id`s are referenced from the overview above.

In [ ]:
# Specify the RecordSet you want to extract (update as new sets appear in newer versions)
record_set_ids = [rs.id for rs in metadata.record_sets]
dataframes = {}
for rs_id in record_set_ids:
    # Load the records for this RecordSet
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    print(f"Loaded {len(df)} records for RecordSet {rs_id}.")
    dataframes[rs_id] = df

# For illustration, work with the first RecordSet (update if you need a specific one)
if record_set_ids:
    main_record_set_id = record_set_ids[0]
    print(f"\nColumns in RecordSet {main_record_set_id}:")
    print(dataframes[main_record_set_id].columns.tolist())
    dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)

Demonstrate processing steps using a numeric field from the selected DataFrame. All operations refer explicitly to field names by their actual `@id` (column name).

In [ ]:
# List columns to select a numeric field
df = dataframes[main_record_set_id]

print(f"Available columns: {df.columns.tolist()}")

# Select a likely numeric field (replace with a valid field for your analysis)
# For demonstration, let's try to find a column containing 'abundance' or 'MW' (molecular weight)
numeric_candidates = [col for col in df.columns if any(key in col.lower() for key in ["abundance", "mw", "count", "coverage", "intensity", "peptide"])]
if numeric_candidates:
    numeric_field_id = numeric_candidates[0]  # use first match as example
else:
    # fallback: first numeric dtype column
    numeric_field_id = df.select_dtypes(["number"]).columns[0]

print(f"Selected numeric field for filtering: {numeric_field_id}")

threshold = df[numeric_field_id].median() if pd.api.types.is_numeric_dtype(df[numeric_field_id]) else 10

# Filter records with the numeric field greater than threshold (median or 10)
filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold}: {len(filtered_df)} records.")
print(filtered_df[[numeric_field_id]].head())

# Normalize the selected numeric field
filtered_df = filtered_df.copy()
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"\nNormalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Try grouping by a categorical field (e.g., description, accession); use first non-numeric field
cat_candidates = [col for col in df.columns if df[col].dtype == object and col != numeric_field_id]
group_field = cat_candidates[0] if cat_candidates else None
if group_field:
    grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
    print(f"\nGrouped data by {group_field} (mean {numeric_field_id}):")
    print(grouped_df.head())

## 5. Visualization

Let's visualize the distribution of the selected numeric field as a histogram, and if possible, a boxplot grouped by a category.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid")

# Histogram of the selected numeric field
plt.figure(figsize=(8,4))
sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.show()

# If grouping category available, draw a boxplot
if group_field:
    # Plot only for top 10 categories by frequency to avoid clutter
    top_cats = df[group_field].value_counts().index[:10]
    plt.figure(figsize=(12, 5))
    sns.boxplot(x=group_field, y=numeric_field_id, data=df[df[group_field].isin(top_cats)])
    plt.title(f"{numeric_field_id} by {group_field} (Top 10 categories)")
    plt.xticks(rotation=45, ha='right')
    plt.show()

## 6. Conclusion

- We successfully parsed the metadata and contents of the FAIR\^2 mass spectrometry dataset with `mlcroissant`.
- Using entity `@id`s, we demonstrated how to enumerate available record sets and fields, extract data frames, and carry out basic EDA steps: threshold filtering, normalization, grouping, and visualization.
- This notebook can be extended for feature engineering, deeper biological analysis, or custom data science workflows depending on the research question. Be sure to use the `@id` references for robust and reproducible access to all fields and sets in the Croissant schema.